Data Transormation part-1

Topics to be covered:
1) Reading CSV Data
2) Understanding DataFrames
3) Common Transformations
4) Data Cleaning
5) Aggregations
6) Window Functions
7) Joins
8) Writing Transformed Data

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [ ]:
# Create SparkSession:

spark = SparkSession.builder.appName("CSV Transformation").getOrCreate()

In [ ]:
# Read CSV:
df = spark.read.csv(
    "dataFiles/employees.csv",
    header=True,
    inferSchema=True
    )


In [ ]:
# View Data:
df.show()

In [ ]:
# Check Schema:
df.printSchema()

In [ ]:
# Selecting Specific Columns: 
df.select("name", "salary").show()

In [ ]:
# Other way to Select and show specific columns:
df.select(
    col("name"),
    col("salary")
).show()

In [ ]:
# Rename Columns:
df.select(
    col("salary").alias("annual_salary")
).show()

In [ ]:
# Filtering Data:

# Employees with salary > 60000:

df.filter(df.salary > 60000).show()

In [ ]:
# Employees with salary > 60000:
df.where("salary > 60000").show()

In [ ]:
# Multiple Conditions:
# Get the Employees whose salary is above 50000 and dept is IT.

df.filter(
    (df.salary > 50000) &
    (df.department == "IT")
).show()

In [ ]:
# Adding new column: 

# Increase the salary of employees by 10% :

df = df.withColumn(
    "new_salary",
    col("salary") * 1.10
)

df.show()

In [ ]:
# String Transformation:

df.withColumn(
    "name_upper",
    upper(col("name"))
).show()

In [ ]:
df.withColumn(
    "name_lower",
    lower(col("name"))
).show()

In [ ]:
# Trim Spaces:
df.withColumn(
    "name_trimmed",
    trim(col("name"))
).show()

In [ ]:
df.withColumn(
    "emp_info",
    concat(col("name"), lit("-"), col("department"))
).show()

In [ ]:
spark.stop()

In [ ]:
spark = SparkSession.builder.appName("csv_transformation2").getOrCreate()

df1 = spark.read.csv(
    "dataFiles/employees.csv",
    header=True,
    inferSchema=True
)

df1.show()

In [ ]:
# Check Nulls:

df1.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df1.columns
]).show()

In [ ]:
# Drop null rows:
df1.na.drop().show()

In [ ]:
df1.na.fill({
    "name": "Unkown",
    "department": "Unkown",
    "age": 0
}).show()

In [ ]:
df1.show()

In [ ]:
# Remove Duplicates:

# df1.dropDuplicates(["id"]).show()
# df1.dropDuplicates(["department"]).show()


In [ ]:
# Type Conversion:

# Convert salary to double
df2 = df1.withColumn(
    "salary",
    col("salary").cast(DoubleType())
)

df1.printSchema()
df2.printSchema()
df2.show()

In [ ]:
# Sorting Data 
df2.orderBy("salary").show()
df2.orderBy("age").show()

In [ ]:
# Order by multiple columns 

df2.orderBy(
    col("department"),
    col("salary").desc()
).show()

In [ ]:
# Group By and Aggregation:

# Departmentwise average Salary:

df2.groupBy("department").avg("salary").show()

In [ ]:
# Multiple Aggregations:

df2.groupBy("department").agg(
    avg("salary").alias("avg_salary"),
    max("salary").alias("max_salary"),
    min("salary").alias("min_salary"),
    count("*").alias("employee_count")
).show()

In [ ]:
# Window Functions:

window_spec = Window.partitionBy(
    "department"
).orderBy(
    col("salary").desc()
)

# window_spec

df2.withColumn(
    "rank",
    rank().over(window_spec)
).show()

In [ ]:
# Join CSV Data:
emp_df = spark.read.csv(
    "dataFiles/emp.csv",
    header=True,
    inferSchema=True
)


dept_df = spark.read.csv(
    "dataFiles/dept.csv",
    header=True,
    inferSchema=True
)

emp_df.show()
dept_df.show()

In [ ]:
# Inner Join:

result_inner = emp_df.join(
    dept_df,
    "id",
    "inner"
)

result_inner.show()

In [ ]:
# Left Join:

result_left = emp_df.join(
    dept_df,
    "id",
    "left"
)

result_left.show()

In [ ]:
result_right = emp_df.join(
    dept_df,
    "id",
    "right"
)

result_right.show()

In [ ]:
result_full = emp_df.join(
    dept_df,
    "id",
    "full"
)

result_full.show()

In [ ]:
# Using SQL on CSV Data:
# Create temperory Views

df2.createOrReplaceTempView(
    "employ"
)

res = spark.sql("""
    SELECT
        department,
        AVG(salary) avg_salary
        FROM employ
        Group By department
""")

res.show()

In [ ]:
res1 = spark.sql("""
        select
                 *
        from
                 employ
""")

res1.show()